In [0]:
from pyspark.sql import functions as F

silver_sales_df = spark.table(
    "workspace.default.silver_sales"
)

silver_products_df = spark.table(
    "workspace.default.silver_products"
)

print(f"Silver sales rows: {silver_sales_df.count():,}")
print(f"Silver product rows: {silver_products_df.count():,}")

In [0]:
gold_revenue_df = (
    silver_sales_df.alias("sales")
    .join(
        silver_products_df.alias("products"),
        F.col("sales.product_id") == F.col("products.product_id"),
        "left"
    )
    .select(
        F.col("sales.order_id"),
        F.col("sales.sales_id"),
        F.col("sales.product_id"),
        F.col("sales.customer_id"),
        F.col("sales.ordered_at"),
        F.to_date(F.col("sales.ordered_at")).alias("order_date"),
        F.col("sales.quantity"),
        F.col("sales.unit_price").alias("sales_unit_price"),
        F.col("sales.total_amount").alias("source_total_amount"),
        F.col("products.product_name"),
        F.coalesce(
            F.col("products.category"),
            F.lit("Unknown")
        ).alias("category"),
        F.col("products.price").alias("product_price"),
        F.col("products.price_level"),
        F.col("products.rating"),
        F.round(
            F.col("sales.quantity") * F.col("products.price"),
            2
        ).alias("revenue"),
        F.col("sales.ingested_at")
    )
)

display(gold_revenue_df.limit(10))

In [0]:
gold_category_df = (
    gold_revenue_df
    .groupBy("category")
    .agg(
        F.countDistinct("order_id").alias("total_orders"),
        F.countDistinct("product_id").alias("distinct_products"),
        F.countDistinct("customer_id").alias("distinct_customers"),
        F.sum("quantity").alias("units_sold"),
        F.round(F.sum("revenue"), 2).alias("total_revenue"),
        F.round(F.avg("revenue"), 2).alias("average_order_revenue")
    )
    .orderBy(F.col("total_revenue").desc())
)

display(gold_category_df)

In [0]:
gold_monthly_df = (
    gold_revenue_df
    .withColumn(
        "order_month",
        F.date_trunc("month", F.col("ordered_at"))
    )
    .groupBy("order_month")
    .agg(
        F.countDistinct("order_id").alias("total_orders"),
        F.countDistinct("customer_id").alias("distinct_customers"),
        F.sum("quantity").alias("units_sold"),
        F.round(F.sum("revenue"), 2).alias("total_revenue")
    )
    .orderBy("order_month")
)

display(gold_monthly_df)

In [0]:
(
    gold_revenue_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.default.gold_fct_revenue")
)

(
    gold_category_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.default.gold_sales_by_category")
)

(
    gold_monthly_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.default.gold_monthly_revenue")
)

print("Created workspace.default.gold_fct_revenue")
print("Created workspace.default.gold_sales_by_category")
print("Created workspace.default.gold_monthly_revenue")

In [0]:
gold_validation_df = spark.sql("""
    SELECT
        COUNT(*) AS fact_rows,
        COUNT(DISTINCT order_id) AS distinct_orders,
        SUM(
            CASE
                WHEN product_name IS NULL THEN 1
                ELSE 0
            END
        ) AS unmatched_product_rows,
        ROUND(SUM(revenue), 2) AS total_revenue
    FROM workspace.default.gold_fct_revenue
""")

display(gold_validation_df)